# Lexical Filter-based Feature Selection

In [ ]:
import pandas as pd
from mlxtend.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import filter_feature_selection
import scipy.stats as stats

The URL dataset with generated lexical features are read from a csv file. These are then displayed for visualization purposes.

In [ ]:
dataset = pd.read_csv('final_unbalanced_with_lexical.csv')

dataset.head()

In [ ]:
features = dataset.iloc[:, 1:]
url_type = dataset.iloc[:, 0]

In [ ]:
features.head()

In [ ]:
url_type.head()

The respective F-score of the features are then calculated using ANOVA. The higher the F-score, the better a feature is.

In [ ]:
anova_features, anova_f_scores, anova_p = filter_feature_selection.anova_feature_selection('final_unbalanced_with_lexical.csv')

In [ ]:
anova_p

In [ ]:
anova_f_scores = [i for i in anova_f_scores if str(i) != 'nan']
anova_f_scores = np.array(anova_f_scores)

anova_f_scores

Using the computed F-scores, the Z-scores are obtained so that the features with F-scores belonging in the upper 50 percentile are selected.

In [ ]:
anova_z_scores = stats.zscore(anova_f_scores)

In [ ]:
percentile = []
for z_score in anova_z_scores:
    percentile.append(stats.norm.cdf(z_score))

In [ ]:
feature_percentile_dict = {}
print('Feature Percentile: ')
for i in range(len(percentile)):
    print(anova_features[i], ': ', percentile[i])
    if percentile[i] > 0.50:
        feature_percentile_dict[anova_features[i]] = percentile[i]

Shown below are the features with F-scores that belong in the upper 50 percentile.

In [ ]:
feature_percentile_dict

Chi2 is used to determine the significance of categorical features. The higher the Chi2 statistics, the better a feature is and the lower its P value, the more significant the observation. Based on the results, both categorical features are to be retained.

In [ ]:
filter_feature_selection.chi2_feature_selection('final_unbalanced_with_lexical.csv')

Features with high correlation are to be weeded out. This is to eliminate redundancy in the featureset.

In [ ]:
features_high_correlation = filter_feature_selection.correlation_feature_selection('final_unbalanced_with_lexical.csv')

features_high_correlation

In [ ]:
to_remove = []

for feature in feature_percentile_dict:
    for high_corr_group in features_high_correlation:
        if feature in high_corr_group:
            for ref_feature in high_corr_group:
                if (ref_feature != feature) and (ref_feature in feature_percentile_dict.keys()):
                    if feature_percentile_dict[feature] >= feature_percentile_dict[ref_feature]:
                        print('Removed: ', ref_feature)
                        to_remove.append(ref_feature)
                        high_corr_group.remove(ref_feature)
                        #feature_percentile_dict.pop(ref_feature, None)
                    else:
                        print('Removed: ', feature)
                        to_remove.append(feature)
                        high_corr_group.remove(feature)
                        #feature_percentile_dict.pop(feature, None)

print(to_remove)

In [ ]:
for feature in to_remove:
    feature_percentile_dict.pop(feature, None)

In [ ]:
final_anova_features = [key for key in feature_percentile_dict.keys()]

final_anova_features

After combining the results of ANOVA and Chi2 and removing the highly correlated features, the final featureset is seen below.

In [ ]:
final_features = final_anova_features + ['get_tld'] + ['url_scheme'] # ADD CATEGORICAL IF GOOD CHI2 SCORE AND SIGNIFICANT BASED ON P VALUE | WILL AUTOMATE LATER

final_features